# Final smoke — 10 frames: clean vs occluded, near/mid/far

Purpose: side-by-side visual before the overnight RunPod run. Two passes:

- **Pass A — clean (5 frames):** `occluders.ratio=0.0`. One frame per distance (150 / 175 / 200 / 250 / 300 m). No trees, tank silhouette full.
- **Pass B — occluded (5 frames):** `occluders.ratio=1.0`, `count=10-18` bushes @ 1.5-20 m. One frame per distance.

Uses the new altitude filter (`altitude_range_m: [150, 300]`) — no more "camera 30 m underground" combos.

`v1_tank_fov92`, 1280×720, Blender only (Stages 1-2). Qwen skipped.

## 1. Attach datasets

In [ ]:
from pathlib import Path

assets_root = next(iter(Path('/kaggle/input').rglob('hdri'))).parent
code_root = next(iter(Path('/kaggle/input').rglob('engine'))).parent
print('assets:', assets_root)
print('code  :', code_root)
for p in sorted(assets_root.iterdir()):
    print(' assets/', p.name)
for p in sorted(code_root.iterdir()):
    print(' code/', p.name)

## 2. Install BlenderProc + deps

In [ ]:
!pip install -q blenderproc==2.8.0 'numpy<2' 'opencv-python<4.11' pyyaml pillow
# BlenderProc bootstraps Blender 4.x on first run — takes ~2 min.
!blenderproc quickstart 2>&1 | tail -20 || true

## 3. Materialise repo layout in `/kaggle/working`

In [ ]:
import shutil, yaml, copy
from pathlib import Path

WORK = Path('/kaggle/working/repo')
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)

# Copy code (engine, pipelines, configs)
for sub in ('engine', 'pipelines', 'configs'):
    shutil.copytree(code_root / sub, WORK / sub)

# Copy assets
(WORK / 'assets').mkdir()
for sub in ('hdri', 'textures', 'models', 'props'):
    src = assets_root / sub
    if src.exists():
        shutil.copytree(src, WORK / 'assets' / sub)

# Base config → derive two: clean (ratio=0) and occluded (ratio=1).
base_cfg_path = WORK / 'configs' / 'v1_tank_fov92.yaml'
base = yaml.safe_load(base_cfg_path.read_text())

clean = copy.deepcopy(base)
clean['occluders']['enabled'] = False   # no bushes at all
clean_path = WORK / 'configs' / 'smoke_clean.yaml'
clean_path.write_text(yaml.safe_dump(clean, sort_keys=False))

occ = copy.deepcopy(base)
occ['occluders']['enabled'] = True
occ['occluders']['ratio'] = 1.0
occ['occluders']['count_range'] = [10, 18]
occ['occluders']['radius_range'] = [1.5, 20.0]
occ['occluders']['scale_range'] = [0.7, 1.5]
occ_path = WORK / 'configs' / 'smoke_occluded.yaml'
occ_path.write_text(yaml.safe_dump(occ, sort_keys=False))

print('two configs staged:')
print(f'  clean    : {clean_path.name}  (occluders disabled)')
print(f'  occluded : {occ_path.name}  (ratio=1.0, 10-18 bushes)')
print(f'\nCommon camera:')
print(f'  distance_m       : {base["camera"]["distance_m"]}')
print(f'  view_angle_deg   : {base["camera"]["view_angle_deg"]}')
print(f'  altitude_range_m : {base["camera"].get("altitude_range_m")}')
print(f'  hfov_deg         : {base["camera"]["hfov_deg"]}')

## 4. Render — pass A (clean) + pass B (occluded), 5 frames each

In [ ]:
%cd /kaggle/working/repo
# Pass A — clean. n=5 with 5 distances → sample_stratified guarantees one frame per distance.
!MPLBACKEND=Agg blenderproc run engine/render_runner.py \
    --config configs/smoke_clean.yaml \
    --n 5 \
    --out /kaggle/working/out_clean \
    --assets-root assets \
    --seed 42 2>&1 | tail -50

# Pass B — occluded. Different seed so bushes randomize independently.
!MPLBACKEND=Agg blenderproc run engine/render_runner.py \
    --config configs/smoke_occluded.yaml \
    --n 5 \
    --out /kaggle/working/out_occluded \
    --assets-root assets \
    --seed 42 2>&1 | tail -50

## 5. Inspect — clean vs occluded side-by-side per distance

In [ ]:
import json
from pathlib import Path
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

def load_frame(root: Path, stem: str):
    rgb = np.array(Image.open(root / 'images' / f'{stem}.jpg'))
    mask = np.array(Image.open(root / 'vehicle_masks' / f'{stem}.png'))
    lbl_txt = (root / 'labels' / f'{stem}.txt').read_text().strip()
    meta = json.loads((root / 'metadata' / f'{stem}.json').read_text())
    return rgb, mask, lbl_txt, meta

def draw(rgb, mask, lbl_txt):
    h, w = rgb.shape[:2]
    canvas = rgb.copy()
    for line in lbl_txt.splitlines():
        _, cx, cy, bw, bh = line.split()
        cx, cy, bw, bh = map(float, (cx, cy, bw, bh))
        x1 = int((cx - bw/2) * w); y1 = int((cy - bh/2) * h)
        x2 = int((cx + bw/2) * w); y2 = int((cy + bh/2) * h)
        cv2.rectangle(canvas, (x1, y1), (x2, y2), (0, 255, 0), 3)
    binm = (mask > 0).astype(np.uint8) * 255
    ct, _ = cv2.findContours(binm, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(canvas, ct, -1, (0, 200, 255), 2)
    return canvas, int(binm.sum() / 255)

def collect(root: Path):
    """Return {distance: (stem, rgb, mask, lbl, meta)} sorted by distance."""
    out = {}
    for stem_path in sorted((root / 'images').glob('*.jpg')):
        stem = stem_path.stem
        rgb, mask, lbl, meta = load_frame(root, stem)
        out.setdefault(meta['distance_m'], (stem, rgb, mask, lbl, meta))
    return out

CLEAN = Path('/kaggle/working/out_clean')
OCC   = Path('/kaggle/working/out_occluded')
clean_by_d = collect(CLEAN)
occ_by_d   = collect(OCC)

for d in sorted(set(clean_by_d) | set(occ_by_d)):
    fig, ax = plt.subplots(1, 2, figsize=(20, 5.7))
    for i, (label, bag) in enumerate([('CLEAN', clean_by_d), ('OCCLUDED', occ_by_d)]):
        if d not in bag:
            ax[i].set_title(f'{label} {d} m — no frame')
            ax[i].axis('off')
            continue
        stem, rgb, mask, lbl, meta = bag[d]
        canvas, mask_px = draw(rgb, mask, lbl)
        n_boxes = len([l for l in lbl.splitlines() if l.strip()])
        ax[i].imshow(canvas)
        ax[i].set_title(f'{label}  d={d}m  angle={meta["view_angle_deg"]}°  '
                        f'{meta["season"]}/{meta["landscape"]}  '
                        f'mask={mask_px}px  boxes={n_boxes}',
                        fontsize=11)
        ax[i].axis('off')
    plt.tight_layout(); plt.show()

# Summary table
print('\n=== summary ===')
print(f"{'d_m':>5} {'clean_mask':>11} {'occ_mask':>11} {'ratio':>7} {'clean_ang':>10} {'occ_ang':>10}")
for d in sorted(set(clean_by_d) | set(occ_by_d)):
    c = clean_by_d.get(d); o = occ_by_d.get(d)
    if c is None or o is None:
        continue
    c_mask = int((c[2] > 0).sum()); o_mask = int((o[2] > 0).sum())
    ratio = o_mask / c_mask if c_mask > 0 else 0
    print(f'{d:>5} {c_mask:>11} {o_mask:>11} {ratio*100:>6.0f}% {c[4]["view_angle_deg"]:>9}° {o[4]["view_angle_deg"]:>9}°')